In [ ]:
import gzip
import glob
import csv
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

aa=['GLYD', 'SCA', 'SCV', 'SCL', 'SCI', 'SCP', 'SCHD', 'SCHE', 'SCHP', 'SCW', 'SCY', 'SCYM', 'SCM',
    'SCC', 'SCCM', 'SCF', 'SCS', 'SCT', 'SCK', 'SCKN', 'SCR', 'SCRN', 'SCE', 'SCEN', 'SCD',
    'SCDN','SCQ', 'SCN']

PMF='../data/pmf_data/total/'
save_dir='../plot/'

In [ ]:
def free_energy(pKa, charge):
    R=8.314*10**(-3) #kJ/mol.K
    T=302.3545798089109 #K
    pH=7.0
    return -charge*R*T*np.log(10)*(pKa-pH)

def get_ref_per_acid(acid):
    #source : Platzer et al. (2014)
    #pKa : ARG 13.9, LYS 10.34, ASP 3.86, GLU 4.34, HIS 6.45, CYS 8.49, TYR 9.76
    neutrals=['SCHP', 'SCKN', 'SCRN', 'SCDN', 'SCEN', 'SCCM', 'SCYM']
    pKas=[6.45, 10.34, 13.9, 3.86, 4.34, 8.49, 9.76]
    charges=[1, -1, -1, 1, 1, -1, -1]
    if acid in neutrals:
        zero_ref=free_energy(pKas[neutrals.index(acid)], charges[neutrals.index(acid)])
    else:
        zero_ref=0
    return zero_ref

def PMF_residue(acid):
    dfpmf=pd.read_table(PMF+f'{acid}/pmf_{acid}.dat'.lower(), sep="\s+")
    return dfpmf



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec

# Configuration pour article scientifique
plt.rcParams.update({
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 7,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 5,
    'lines.linewidth': 0.8,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'errorbar.capsize': 1,
})

# Largeur figure: 8.26 cm = 3.252 inches
fig_width = 8.26 / 2.54  # conversion cm -> inches

# Positions personnalisées pour la grille verticale
custom_vgrid = [9.5, 19.5, 30]


def plot_independent_graphs(acid_list, label_list, x_grid_interval=10, y_grid_interval=10):
    """
    Affiche des graphiques indépendants pour les PMF (ou profils de densité moyennés) d'acides.
    """
    for group_acids, group_labels in zip(acid_list, label_list):
        fig, ax = plt.subplots(figsize=(fig_width, 1.5))
        for acid, label in zip(group_acids, group_labels):
            data = PMF_residue(acid)
            x, y, e = data['z'], data['mean'], data['se']
            ax.errorbar(x, y, e, label=label, lw=0.8, capsize=1, capthick=0.5, elinewidth=0.5)
        
        ax.xaxis.set_major_locator(ticker.MultipleLocator(x_grid_interval))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(y_grid_interval))
        # Grille verticale personnalisée
        for xv in custom_vgrid:
            ax.axvline(x=xv, linestyle='--', alpha=0.8, linewidth=0.6, color='gray', zorder=0)
        ax.yaxis.grid(True, which='major', linestyle='--', alpha=0.5, linewidth=0.3)
        ax.set_xlim(0, 35)
        ax.set_xlabel("z (Å)")
        ax.set_ylabel("PMF (kJ/mol)")
        ax.legend(loc='upper right', frameon=False, 
                  borderpad=0.4, handlelength=1.5, handletextpad=0.5, labelspacing=0.3,
                  ncols=len(group_labels))
        
        plt.savefig(save_dir+'{}.png'.format(label), bbox_inches='tight', dpi=600, transparent=True)
        plt.show()


def plot_combined_graphs(acid_list, label_list, name, titrable=False, height_per_unit=0.03, x_grid_interval=10, y_grid_interval=10):
    """
    Affiche des graphiques combinés en une seule colonne pour les PMF.
    Optimisé pour article scientifique (largeur 8.26cm, 600dpi).
    - Légende directement sur le graphique (fond blanc, sans encadré)
    - Échelle Y uniforme (même ratio pixels/unité) pour tous les graphiques
    """
    n_plots = len(acid_list)

    # Définir les limites Y pour chaque groupe et calculer les plages
    y_limits = []
    for group_labels in label_list:
        if 'CYS' in group_labels or 'ALA' in group_labels:
            y_limits.append((-25, 14))
        elif 'THR' in group_labels:
            y_limits.append((-3, 23))
        elif 'PHE' in group_labels:
            y_limits.append((-23, 13))
        elif 'PRO' in group_labels:
            y_limits.append((-8, 23))
        elif 'TYR$^0$' in group_labels:
            y_limits.append((-18, 38))
        elif 'LYS$^0$' in group_labels:
            y_limits.append((-6, 33))
        elif 'CYS$^0$' in group_labels:
            y_limits.append((-13, 30))
        elif 'ASP$^0$' in group_labels:
            y_limits.append((-3, 38))
        elif 'GLU$^0$' in group_labels:
            y_limits.append((-3, 33))
        elif 'HSP$^+$' in group_labels:
            y_limits.append((-3, 25))
        elif 'ARG$^0$' in group_labels:
            y_limits.append((-8, 58))
        else:
            y_limits.append((-25, 25))  # défaut

    # Calculer les plages Y
    y_ranges = [ymax - ymin for ymin, ymax in y_limits]
    
    # Hauteur de base par unité Y (pour avoir la même échelle)
    # height_per_unit = 0.03  # inches par unité Y
    
    # Créer les ratios de hauteur proportionnels aux plages Y
    height_ratios = [r * height_per_unit for r in y_ranges]
    
    total_height = sum(height_ratios) + 0.5  # marge pour xlabel
    
    # Créer la figure avec GridSpec
    fig = plt.figure(figsize=(fig_width, total_height))
    gs = GridSpec(n_plots, 1, figure=fig, height_ratios=height_ratios, hspace=0)
    
    # Créer les axes
    plot_axes = [fig.add_subplot(gs[i, 0]) for i in range(n_plots)]
    
    for idx, (ax, group_acids, group_labels) in enumerate(zip(plot_axes, acid_list, label_list)):
        for acid, label in zip(group_acids, group_labels):
            data = PMF_residue(acid)
            if not titrable:
                x, y, e = data['z'], data['mean'], data['se']
            else:
                correction = get_ref_per_acid(acid)
                x, y, e = data['z'], data['mean'] + correction, data['se']
            ax.errorbar(x, y, e, label=label, lw=0.8, capsize=1, capthick=0.4, elinewidth=0.4)
        
        # Configuration du graphique
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
        
        ax.xaxis.set_major_locator(ticker.MultipleLocator(x_grid_interval))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(y_grid_interval))
        # Grille verticale personnalisée
        for xv in custom_vgrid:
            ax.axvline(x=xv, linestyle='--', alpha=0.8, linewidth=0.8, color='gray', zorder=0)
        ax.yaxis.grid(True, which='major', linestyle='--', alpha=0.5, linewidth=0.3)
        ax.set_xlim(0, 35)
        ax.set_ylim(y_limits[idx])
        
        # Légende directement sur le graphique (fond blanc, sans encadré)
        ax.legend(loc='upper right', ncols=len(group_labels), fontsize=6,
                  frameon=True, facecolor='white', edgecolor='none',
                  handlelength=1.5, handletextpad=0.4, columnspacing=1.0)
        
        ax.tick_params(axis='both', which='major', pad=2)
        
        # Cacher les labels x sauf pour le dernier
        if idx < n_plots - 1:
            ax.tick_params(labelbottom=False)
    
    # Ajout des chiffres romains au-dessus du premier subplot
    roman_numerals = [('I', 4.75), ('II', 14.5), ('III', 24.75), ('IV', 32.5)]
    for numeral, x_pos in roman_numerals:
        plot_axes[0].text(x_pos, 1.02, numeral, transform=plot_axes[0].get_xaxis_transform(),
                         ha='center', va='bottom', fontsize=7, fontweight='normal')
    
    plot_axes[-1].set_xlabel("z (Å)")
    # Label Y commun
    fig.text(0.02, 0.5, "PMF (kJ/mol)", va='center', rotation='vertical')
    
    plt.savefig(save_dir+'{}.png'.format(name), bbox_inches='tight', dpi=600, transparent=True)
    plt.show()


g1 = ['SCA', 'SCV', 'SCL', 'SCI']
g2 = ['SCC', 'SCM']
g3 = ['SCS', 'SCT', 'SCN', 'SCQ']
g4 = ['GLYD', 'SCP']
g5 = ['SCF', 'SCY', 'SCW']

l1 = ['ALA', 'VAL', 'LEU', 'ILE']
l2 = ['CYS', 'MET']
l3 = ['SER', 'THR', 'ASN', 'GLN']
l4 = ['GLYD', 'PRO']
l5 = ['PHE', 'TYR', 'TRP']

test1 = [g1, g2, g3, g5]
test2 = [g1, g2, g3, g5, g4]

label1 = [l1, l2, l3, l5]
label2 = [l1, l2, l3, l5, l4]

plot_combined_graphs(test2, label2, '5_graphics-1', titrable=False, height_per_unit=0.028)

In [ ]:
g1=['SCR', 'SCRN']
g2=['SCD', 'SCDN']
g3=['SCK', 'SCKN']
g4=['SCE', 'SCEN']
g5=['SCHE', 'SCHD', 'SCHP']
g6=['SCC', 'SCCM']
g7=['SCY', 'SCYM']
g8=['GLYD', 'SCP']

l1=['ARG$^+$', 'ARG$^0$']
l2=['ASP$^-$', 'ASP$^0$']
l3=['LYS$^+$', 'LYS$^0$']
l4=['GLU$^-$', 'GLU$^0$']
l5=['HSE$^0$', 'HSD$^0$', 'HSP$^+$']
l6=['CYS$^0$', 'CYS$^-$']
l7=['TYR$^0$', 'TYR$^-$']
l8=['GLYD', 'PRO']

test1 = [g6, g7, g2, g4, g3, g1, g5]

label1 = [l6, l7, l2, l4, l3, l1, l5]


plot_combined_graphs(test1, label1, '8_graphics_1col-1', titrable=True, height_per_unit=0.025)
